# 长期记忆-agent
## 1、在工具中访问长期记忆
### 1.1、基于InmemoryStore

In [5]:
import os
from typing import Callable
from langgraph.store.memory import InMemoryStore
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import wrap_model_call
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_core.stores import InMemoryStore
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.prebuilt import ToolRuntime

load_dotenv(override=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")
model =init_chat_model(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={
        "thinking": {"type": "disabled"}  # 关键：在这里关闭思考模式
    }
)

In [7]:
from langchain.agents import create_agent
from langgraph.prebuilt import ToolRuntime
from langgraph.store.memory import InMemoryStore
from typing import NotRequired
store  = InMemoryStore()
# 自定义一个继承于Agentstate的类
class CustomState(AgentState):
    user_id: NotRequired[str]

@tool(parse_docstring=True)
def save_user_info(name:str,runtime:ToolRuntime)->str:
    """
    将客户信息保存在长期记忆中

    Args:
        name:用户名
        runtime:工具的运行时

    Returns:
        str:保存状态
    """
    namespace = ("users",)
    key = runtime.state["user_id"]
    value = {"name":name}
    runtime.store.put(namespace, key, value)
    return"saved"

@tool(parse_docstring=True)
def get_user_info(runtime:ToolRuntime)->str:
    """
    从长期记忆中读取客户信息哦
    Args:
        runtime:工具运行时长

    Returns:
        str:用户信息
    """
    namespace = ("users",)
    key = runtime.state["user_id"]
    item = runtime.store.get(namespace, key)
    return str(item.value) if item else "unknown"
agent  = create_agent(
    model=model,
    tools =[save_user_info, get_user_info],
    store =store,
    state_schema = CustomState,
    system_prompt = "用户提交个人信息时，可以实用工具保护用户信息，如果用户询问个人信息，可以尝试使用工具读取用户信息"
)


print("=" * 30, "-> 第一个会话（线程） <-", "=" * 30)

response1 = agent.invoke({
    "messages": [HumanMessage("你好，很高兴认识你，我是小花")],
    "user_id": "user-1"
})

for msg in response1["messages"]:
    msg.pretty_print()

print("=" * 30, "-> 第二个会话（线程） <-", "=" * 30)

response2 = agent.invoke({
    "messages": [HumanMessage("我是谁")],
    "user_id": "user-1"
})

for msg in response2["messages"]:
    msg.pretty_print()

============================== -> 第一个会话（线程） <- ==============================
================================ Human Message =================================

你好，很高兴认识你，我是小花
================================== Ai Message ==================================

你好，小花，很高兴认识你！😊

请问有什么我可以帮你的吗？
============================== -> 第二个会话（线程） <- ==============================
================================ Human Message =================================

我是谁
================================== Ai Message ==================================

我来帮您查询一下您的信息。
Tool Calls:
  get_user_info (call_00_R5WFzkpBL8BEfmXd7XtL1004)
 Call ID: call_00_R5WFzkpBL8BEfmXd7XtL1004
  Args:
================================= Tool Message =================================
Name: get_user_info

unknown
================================== Ai Message ==================================

抱歉，我这边没有查询到您的个人信息记录，所以暂时无法确认您是谁。

如果您希望我记住您的身份，可以直接告诉我您的名字，我会帮您保存下来，下次就能识别您了。
